# FedCloud client node (Colab / Kaggle) → server on Render

This notebook turns this cloud machine into one federated-learning client.
1. Set `CIDS` below to this node's client id. Use one notebook per client for the cleanest demo, or e.g. `[0, 1]` to run two clients here.
2. **Runtime → Run all.** On Kaggle, switch **Settings → Internet → On** first.

The node downloads the code and **only its own** `client_k.npz`. It trains locally and sends only model weights (17 kB) to the server. Raw transactions never leave this machine.

In [ ]:
SERVER = "https://fedcloud-server.onrender.com"   # Render URL of the aggregation server
CIDS = [0]                                          # this node's client id(s), 0-4
REPO = "https://raw.githubusercontent.com/deepankitasaha/cc/main/fedcloud-fl"


In [ ]:
import os, urllib.request
os.makedirs("fedcloud", exist_ok=True)
for f in ["__init__", "async_client", "data", "metrics", "model", "netemu", "serialization"]:
    urllib.request.urlretrieve(f"{REPO}/fedcloud/{f}.py", f"fedcloud/{f}.py")
DATA = {}
for c in CIDS:
    DATA[c] = f"client_{c}.npz"
    urllib.request.urlretrieve(f"{REPO}/data/parts_dir0.5/client_{c}.npz", DATA[c])
    print(f"client {c}: downloaded its private partition ({os.path.getsize(DATA[c]) / 1e6:.2f} MB)")


In [ ]:
# Wake the server (Render's free tier sleeps when idle) and measure the real network round trip.
import time, socket, platform
for i in range(30):
    try:
        urllib.request.urlopen(SERVER + "/health", timeout=10).read(); break
    except Exception:
        print("waiting for server to wake up..."); time.sleep(5)
rtts = []
for _ in range(5):
    t = time.time(); urllib.request.urlopen(SERVER + "/health", timeout=10).read(); rtts.append((time.time() - t) * 1000)
print(f"this node: {socket.gethostname()} ({platform.node()})")
print(f"server reachable; HTTPS round trip to {SERVER}: median {sorted(rtts)[2]:.0f} ms over the public internet")


In [ ]:
# Start the FedAsync client(s): pull global model -> train on local data -> push weights, until the server says done.
import subprocess, sys, threading
procs = [subprocess.Popen([sys.executable, "-u", "-m", "fedcloud.async_client", "--server", SERVER, "--cid", str(c),
                           "--data", DATA[c], "--no-emu"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
         for c in CIDS]
def pump(p):
    for line in p.stdout:
        print(line, end="", flush=True)
threads = [threading.Thread(target=pump, args=(p,)) for p in procs]
for t in threads: t.start()
for t in threads: t.join()
print("done - open the server dashboard to see the final AUPRC and download the run log")
